# Qwen2.5-VL-7B LoRA Word-Label Fine-Tuning

Notebook này fine-tune **Qwen2.5-VL-7B-Instruct** bằng LoRA/QLoRA cho bài toán phân loại ảnh `real/fake`.

Protocol:

```text
Train: Tiny-GenImage combined train split
Early stopping: Tiny-GenImage train_inner/val_inner
Internal sanity test: Tiny-GenImage validation split
External benchmark: 1000 random streaming samples from CommunityForensics-Eval / CompEval
```

Mục tiêu training không ép `real/fake` là 1 token. Ground truth là text answer ngắn `real` hoặc `fake`, còn evaluation score xác suất chuỗi `P("real")` và `P("fake")`.


## 0. Install Dependencies


In [ ]:
%pip install -q -U "transformers>=4.49.0" accelerate peft bitsandbytes datasets qwen-vl-utils scikit-learn pandas tqdm pillow safetensors


## 1. Imports And Project Root


In [ ]:
import gc
import io
import json
import os
import random
import sys
import time
from pathlib import Path
from typing import Any

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from PIL import Image, ImageFile
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, IterableDataset
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)
from peft import (
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_kbit_training,
    set_peft_model_state_dict,
)


def find_code_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(kaggle_input.glob("*"))
        candidates.extend(kaggle_input.glob("*/*"))
        candidates.extend(kaggle_input.glob("*/*/*"))
        for init_file in kaggle_input.rglob("__init__.py"):
            if init_file.parent.name == "data_loader":
                return init_file.parent.parent

    for candidate in candidates:
        if (candidate / "data_loader" / "__init__.py").exists():
            return candidate

    print("/kaggle/input children:")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.glob("*")):
            print(" -", path)
    raise FileNotFoundError("Cannot find HoangHa_Code/data_loader.")


CODE_ROOT = find_code_root()
PROJECT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else CODE_ROOT
sys.path.insert(0, str(CODE_ROOT))
print("CODE_ROOT =", CODE_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)

from data_loader import (  # noqa: E402
    TinyGenImageKaggleConfig,
    TinyGenImageKaggleDataset,
    UnifiedSample,
    build_kaggle_tiny_index,
    build_kaggle_tiny_splits,
    collate_unified_batch,
    find_tiny_genimage_root,
    summarize_index,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()
PIN_MEMORY = torch.cuda.is_available()
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print("DEVICE =", DEVICE)
print("AMP_DTYPE =", AMP_DTYPE)


## 2. Config


In [ ]:
DATASET_ROOT = None

BASE_MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
COMMFOR_DATASET_NAME = "OwensLab/CommunityForensics-Eval"
COMMFOR_SPLIT = "CompEval"
COMMFOR_STREAMING = True
MAX_COMMFOR_EVAL_SAMPLES = 1000
COMMFOR_SHUFFLE_BUFFER_SIZE = 100

# Qwen fine-tuning is much heavier than CLIP/ResNet. Use 5000 for the first run.
# Set MAX_TRAIN_SAMPLES = None when you are ready to train on full Tiny-GenImage combined.
MAX_TRAIN_SAMPLES = 5000
MAX_TINY_INTERNAL_TEST_SAMPLES = 1000
MAX_VAL_SAMPLES = 1000

BALANCE_REAL = True
RANDOM_SEED = 42
VAL_FRACTION = 0.2

TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
NUM_WORKERS = 0
MAX_EPOCHS = 2
PATIENCE = 1
MIN_DELTA = 1e-3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0

LOAD_IN_4BIT = True
USE_GRADIENT_CHECKPOINTING = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 128 * 28 * 28
NORMALIZE_CANDIDATE_LOGPROB = True

PROMPT_TEMPLATE = (
    "Look at the image and decide whether it is real or AI-generated. "
    "Answer with exactly one word: real or fake."
)
LABEL_ID_TO_TEXT = {0: "real", 1: "fake"}
CANDIDATE_TEXTS = ["real", "fake"]

SAVE_PREDICTIONS = True
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "qwen25vl_lora_word_label"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_ROOT / RUN_ID
for subdir in ["adapter", "metrics", "predictions"]:
    (RUN_DIR / subdir).mkdir(parents=True, exist_ok=True)
print("RUN_ID =", RUN_ID)
print("RUN_DIR =", RUN_DIR)


## 3. Tiny-GenImage Combined Dataloader


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def stratified_train_val_split(df: pd.DataFrame, val_fraction: float, seed: int):
    if len(df) < 4 or val_fraction <= 0:
        return df.reset_index(drop=True), df.reset_index(drop=True)

    stratify = df["label"].astype(str) + "_" + df["generator"].astype(str)
    if stratify.value_counts().min() < 2:
        stratify = df["label"]
    if pd.Series(stratify).value_counts().min() < 2:
        stratify = None

    train_df, val_df = train_test_split(
        df,
        test_size=val_fraction,
        random_state=seed,
        shuffle=True,
        stratify=stratify,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def maybe_limit_df(df: pd.DataFrame, max_samples: int | None, seed: int) -> pd.DataFrame:
    if max_samples is None or len(df) <= max_samples:
        return df.reset_index(drop=True)
    return df.sample(n=max_samples, random_state=seed, replace=False).reset_index(drop=True)


def build_tiny_combined_splits(detected_root: Path) -> dict[str, Any]:
    config = TinyGenImageKaggleConfig(
        dataset_root=str(detected_root),
        eval_case="combined",
        balance_real=BALANCE_REAL,
        seed=RANDOM_SEED,
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_eval_samples=MAX_TINY_INTERNAL_TEST_SAMPLES,
    )
    splits = build_kaggle_tiny_splits(config)
    print(splits["notes"])
    print("train real/fake:", splits["train_real_count"], splits["train_fake_count"])
    print("tiny test real/fake:", splits["eval_real_count"], splits["eval_fake_count"])
    return splits


def build_tiny_loaders(splits: dict[str, Any]):
    train_inner_df, val_inner_df = stratified_train_val_split(
        splits["train_df"],
        val_fraction=VAL_FRACTION,
        seed=RANDOM_SEED,
    )
    val_inner_df = maybe_limit_df(val_inner_df, MAX_VAL_SAMPLES, RANDOM_SEED)
    tiny_test_df = splits["eval_df"].reset_index(drop=True)

    print("inner train/val:", len(train_inner_df), len(val_inner_df))
    print("tiny internal test:", len(tiny_test_df))

    train_dataset = TinyGenImageKaggleDataset(train_inner_df, eval_case="combined", transform=None)
    val_dataset = TinyGenImageKaggleDataset(val_inner_df, eval_case="combined", transform=None)
    tiny_test_dataset = TinyGenImageKaggleDataset(tiny_test_df, eval_case="combined", transform=None)

    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_unified_batch,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_unified_batch,
    )
    tiny_test_loader = DataLoader(
        tiny_test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_unified_batch,
    )
    return train_loader, val_loader, tiny_test_loader, train_inner_df, val_inner_df, tiny_test_df


seed_everything(RANDOM_SEED)
detected_tiny_root = find_tiny_genimage_root(DATASET_ROOT)
print("Detected Tiny-GenImage root:", detected_tiny_root)

index_df = build_kaggle_tiny_index(TinyGenImageKaggleConfig(dataset_root=str(detected_tiny_root)))
summary_path = RUN_DIR / "metrics" / f"tiny_structure_summary_{RUN_ID}.csv"
summarize_index(index_df).to_csv(summary_path, index=False)
print("Tiny structure summary saved:", summary_path)
print("Generators:", sorted(index_df["generator"].unique().tolist()))

TINY_SPLITS = build_tiny_combined_splits(detected_tiny_root)
train_loader, val_loader, tiny_test_loader, train_inner_df, val_inner_df, tiny_test_df = build_tiny_loaders(TINY_SPLITS)
train_inner_df.to_csv(RUN_DIR / "metrics" / "tiny_train_inner_split.csv", index=False)
val_inner_df.to_csv(RUN_DIR / "metrics" / "tiny_val_inner_split.csv", index=False)
tiny_test_df.to_csv(RUN_DIR / "metrics" / "tiny_internal_test_split.csv", index=False)


## 4. CommunityForensics-Eval Streaming Dataloader


In [ ]:
def image_from_commfor_record(record: dict[str, Any]) -> Image.Image:
    image_data = record.get("image_data", record.get("image"))
    if isinstance(image_data, Image.Image):
        return image_data.convert("RGB")
    if isinstance(image_data, (bytes, bytearray)):
        return Image.open(io.BytesIO(image_data)).convert("RGB")
    if isinstance(image_data, dict):
        if image_data.get("bytes") is not None:
            return Image.open(io.BytesIO(image_data["bytes"])).convert("RGB")
        if image_data.get("path") is not None:
            return Image.open(image_data["path"]).convert("RGB")
    if isinstance(image_data, list):
        return Image.open(io.BytesIO(bytes(image_data))).convert("RGB")
    raise TypeError(f"Unsupported image_data type: {type(image_data)}")


class CommunityForensicsEvalIterableDataset(IterableDataset):
    def __init__(self, hf_dataset, eval_case="cross_dataset_commfor_eval"):
        self.hf_dataset = hf_dataset
        self.eval_case = eval_case

    def __iter__(self):
        for index, record in enumerate(self.hf_dataset):
            image = image_from_commfor_record(record)
            label = int(record.get("label"))
            model_name = str(record.get("model_name") or record.get("architecture") or "unknown")
            sample = UnifiedSample(
                sample_id=str(record.get("image_name") or f"commfor_eval:{index}"),
                label=label,
                label_name="fake" if label == 1 else "real",
                dataset_source=COMMFOR_DATASET_NAME,
                generator=model_name,
                split=str(record.get("split") or COMMFOR_SPLIT),
                eval_case=self.eval_case,
                prompt=record.get("prompt"),
                metadata={
                    "image_name": record.get("image_name"),
                    "format": record.get("format"),
                    "resolution": record.get("resolution"),
                    "mode": record.get("mode"),
                    "model_name": record.get("model_name"),
                    "architecture": record.get("architecture"),
                    "real_source": record.get("real_source"),
                    "subset": record.get("subset"),
                    "nsfw_flag": record.get("nsfw_flag"),
                },
            ).as_dict()
            sample["image_name"] = record.get("image_name")
            sample["model_name"] = record.get("model_name")
            sample["architecture"] = record.get("architecture")
            sample["real_source"] = record.get("real_source")
            sample["subset"] = record.get("subset")
            sample["nsfw_flag"] = record.get("nsfw_flag")
            sample["image"] = image
            yield sample


def build_commfor_eval_loader():
    hf_dataset = load_dataset(
        COMMFOR_DATASET_NAME,
        split=COMMFOR_SPLIT,
        streaming=COMMFOR_STREAMING,
    )
    if COMMFOR_STREAMING:
        hf_dataset = hf_dataset.shuffle(
            seed=RANDOM_SEED,
            buffer_size=COMMFOR_SHUFFLE_BUFFER_SIZE,
        )
        if MAX_COMMFOR_EVAL_SAMPLES is not None:
            hf_dataset = hf_dataset.take(MAX_COMMFOR_EVAL_SAMPLES)
    elif MAX_COMMFOR_EVAL_SAMPLES is not None:
        hf_dataset = hf_dataset.shuffle(seed=RANDOM_SEED).select(range(MAX_COMMFOR_EVAL_SAMPLES))

    dataset = CommunityForensicsEvalIterableDataset(hf_dataset)
    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_unified_batch,
    )


## 5. Load Qwen2.5-VL + QLoRA


In [ ]:
def load_qwen_lora_model():
    quantization_config = None
    if LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=AMP_DTYPE,
        )

    processor = AutoProcessor.from_pretrained(
        BASE_MODEL_NAME,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        trust_remote_code=True,
    )
    processor.tokenizer.padding_side = "right"
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=AMP_DTYPE,
        device_map="auto",
        quantization_config=quantization_config,
        trust_remote_code=True,
    )
    model.config.use_cache = False
    if USE_GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    if LOAD_IN_4BIT:
        model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, processor


model, processor = load_qwen_lora_model()
MODEL_DEVICE = next(model.parameters()).device
print("MODEL_DEVICE =", MODEL_DEVICE)

print("Token check:")
for text in CANDIDATE_TEXTS:
    token_ids = processor.tokenizer.encode(text, add_special_tokens=False)
    print(repr(text), token_ids, "num_tokens=", len(token_ids))


## 6. Prompt, Label Masking, And Candidate Scoring


In [ ]:
def make_user_messages(image: Image.Image) -> list[dict[str, Any]]:
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT_TEMPLATE},
            ],
        }
    ]


def make_full_messages(image: Image.Image, answer: str) -> list[dict[str, Any]]:
    return make_user_messages(image) + [
        {
            "role": "assistant",
            "content": [{"type": "text", "text": answer}],
        }
    ]


def move_inputs_to_device(inputs: dict[str, Any]) -> dict[str, Any]:
    return {key: value.to(DEVICE) if torch.is_tensor(value) else value for key, value in inputs.items()}


def build_qwen_train_inputs(batch: dict[str, Any]) -> dict[str, torch.Tensor]:
    images = batch["image"]
    labels = batch["label"].tolist()

    prompt_texts = []
    full_texts = []
    prompt_lengths = []
    for image, label_id in zip(images, labels):
        answer = LABEL_ID_TO_TEXT[int(label_id)]
        prompt_messages = make_user_messages(image)
        full_messages = make_full_messages(image, answer)
        prompt_text = processor.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        full_text = processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        prompt_inputs = processor(
            text=[prompt_text],
            images=[image],
            padding=False,
            return_tensors="pt",
        )
        prompt_lengths.append(prompt_inputs["input_ids"].shape[1])
        prompt_texts.append(prompt_text)
        full_texts.append(full_text)

    inputs = processor(
        text=full_texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )
    target_labels = inputs["input_ids"].clone()
    for row_idx, prompt_len in enumerate(prompt_lengths):
        target_labels[row_idx, :prompt_len] = -100
    target_labels[target_labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = target_labels
    return move_inputs_to_device(inputs)


def sequence_logprob_from_logits(logits: torch.Tensor, labels: torch.Tensor, normalize_by_length: bool) -> torch.Tensor:
    shift_logits = logits[:, :-1, :].float()
    shift_labels = labels[:, 1:]
    mask = shift_labels.ne(-100)
    safe_labels = shift_labels.masked_fill(~mask, 0)
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_scores = log_probs.gather(dim=-1, index=safe_labels.unsqueeze(-1)).squeeze(-1)
    token_scores = token_scores * mask
    scores = token_scores.sum(dim=-1)
    if normalize_by_length:
        lengths = mask.sum(dim=-1).clamp_min(1)
        scores = scores / lengths
    return scores


@torch.no_grad()
def score_candidate_batch(batch: dict[str, Any], candidate_text: str) -> np.ndarray:
    images = batch["image"]
    prompt_lengths = []
    full_texts = []
    for image in images:
        prompt_messages = make_user_messages(image)
        full_messages = make_full_messages(image, candidate_text)
        prompt_text = processor.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        full_text = processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        prompt_inputs = processor(
            text=[prompt_text],
            images=[image],
            padding=False,
            return_tensors="pt",
        )
        prompt_lengths.append(prompt_inputs["input_ids"].shape[1])
        full_texts.append(full_text)

    inputs = processor(
        text=full_texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )
    candidate_labels = inputs["input_ids"].clone()
    for row_idx, prompt_len in enumerate(prompt_lengths):
        candidate_labels[row_idx, :prompt_len] = -100
    candidate_labels[candidate_labels == processor.tokenizer.pad_token_id] = -100
    inputs = move_inputs_to_device(inputs)
    candidate_labels = candidate_labels.to(DEVICE)

    outputs = model(**inputs)
    scores = sequence_logprob_from_logits(
        outputs.logits,
        candidate_labels,
        normalize_by_length=NORMALIZE_CANDIDATE_LOGPROB,
    )
    return scores.detach().cpu().numpy()


## 7. Metrics And Evaluation


In [ ]:
def compute_metrics(y_true, y_prob) -> dict[str, Any]:
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }
    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        metrics["average_precision"] = float(average_precision_score(y_true, y_prob))
    else:
        metrics["roc_auc"] = None
        metrics["average_precision"] = None
    return metrics


def save_json(path: Path, data: dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def evaluate_by_column(pred_df: pd.DataFrame, column: str) -> pd.DataFrame:
    rows = []
    if column not in pred_df.columns:
        return pd.DataFrame(rows)
    for value, part in pred_df.groupby(column):
        if len(part) == 0:
            continue
        metrics = compute_metrics(part["label"].to_numpy(), part["fake_probability"].to_numpy())
        metrics[column] = value
        metrics["num_samples"] = int(len(part))
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values(column) if rows else pd.DataFrame(rows)


@torch.no_grad()
def predict_word_label(loader, dataset_tag: str):
    model.eval()
    all_labels, all_fake_probs, rows = [], [], []
    for batch in tqdm(loader, desc=dataset_tag, leave=False):
        real_scores = score_candidate_batch(batch, "real")
        fake_scores = score_candidate_batch(batch, "fake")
        scores = np.stack([real_scores, fake_scores], axis=1)
        probs = torch.softmax(torch.tensor(scores), dim=-1).numpy()
        fake_probs = probs[:, 1]

        all_labels.extend(batch["label"].numpy().tolist())
        all_fake_probs.extend(fake_probs.tolist())
        rows.extend(batch["metadata"])
    return np.array(all_labels, dtype=int), np.array(all_fake_probs, dtype=float), pd.DataFrame(rows)


def evaluate_and_save(loader, dataset_tag: str):
    y_true, y_prob, meta_df = predict_word_label(loader, dataset_tag=dataset_tag)
    pred_df = meta_df.copy()
    pred_df["label"] = y_true
    pred_df["predicted_label"] = (y_prob >= 0.5).astype(int)
    pred_df["fake_probability"] = y_prob
    pred_df["model_name"] = "qwen25vl_lora_word_label"
    pred_df["dataset_tag"] = dataset_tag

    metrics = compute_metrics(y_true, y_prob)
    metrics.update({
        "model_name": "qwen25vl_lora_word_label",
        "dataset_tag": dataset_tag,
        "num_samples": int(len(pred_df)),
        "threshold": 0.5,
        "candidate_texts": CANDIDATE_TEXTS,
        "normalize_candidate_logprob": NORMALIZE_CANDIDATE_LOGPROB,
    })

    save_json(RUN_DIR / "metrics" / f"{dataset_tag}_overall_metrics.json", metrics)
    by_generator = evaluate_by_column(pred_df, "generator")
    if len(by_generator):
        by_generator.to_csv(RUN_DIR / "metrics" / f"{dataset_tag}_generator_metrics.csv", index=False)
    by_architecture = evaluate_by_column(pred_df, "architecture")
    if len(by_architecture):
        by_architecture.to_csv(RUN_DIR / "metrics" / f"{dataset_tag}_architecture_metrics.csv", index=False)
    if SAVE_PREDICTIONS:
        pred_df.to_csv(RUN_DIR / "predictions" / f"{dataset_tag}_predictions.csv", index=False)
    print(dataset_tag, json.dumps(metrics, ensure_ascii=False, indent=2))
    return metrics


## 8. Train LoRA Adapter


In [ ]:
config_summary = {
    "run_id": RUN_ID,
    "base_model_name": BASE_MODEL_NAME,
    "method": "qwen25vl_lora_word_label",
    "train_dataset": "Tiny-GenImage",
    "train_eval_case": "combined",
    "external_dataset": COMMFOR_DATASET_NAME,
    "external_split": COMMFOR_SPLIT,
    "prompt_template": PROMPT_TEMPLATE,
    "label_id_to_text": LABEL_ID_TO_TEXT,
    "candidate_texts": CANDIDATE_TEXTS,
    "normalize_candidate_logprob": NORMALIZE_CANDIDATE_LOGPROB,
    "balance_real": BALANCE_REAL,
    "random_seed": RANDOM_SEED,
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "max_val_samples": MAX_VAL_SAMPLES,
    "max_tiny_internal_test_samples": MAX_TINY_INTERNAL_TEST_SAMPLES,
    "max_commfor_eval_samples": MAX_COMMFOR_EVAL_SAMPLES,
    "commfor_shuffle_buffer_size": COMMFOR_SHUFFLE_BUFFER_SIZE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "min_delta": MIN_DELTA,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "load_in_4bit": LOAD_IN_4BIT,
    "use_gradient_checkpointing": USE_GRADIENT_CHECKPOINTING,
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
    "device": DEVICE,
    "amp_dtype": str(AMP_DTYPE),
    "tiny_train_rows": int(len(TINY_SPLITS["train_df"])),
    "tiny_eval_rows": int(len(TINY_SPLITS["eval_df"])),
}
save_json(RUN_DIR / "metrics" / "config.json", config_summary)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
best_metric = -float("inf")
best_epoch = 0
best_adapter_state = None
bad_epochs = 0
history = []
global_step = 0

model.train()
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_losses = []
    step = 0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(train_loader, desc=f"train epoch {epoch}/{MAX_EPOCHS}", leave=False)
    for step, batch in enumerate(progress, start=1):
        inputs = build_qwen_train_inputs(batch)
        outputs = model(**inputs)
        loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()

        if step % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        epoch_losses.append(float(loss.detach().cpu()) * GRADIENT_ACCUMULATION_STEPS)
        progress.set_postfix(loss=f"{np.mean(epoch_losses):.4f}")

    if step % GRADIENT_ACCUMULATION_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else 0.0
    y_val, p_val, _ = predict_word_label(val_loader, dataset_tag="tiny_val")
    val_metrics = compute_metrics(y_val, p_val)
    current = val_metrics["balanced_accuracy"]
    history_row = {"epoch": epoch, "global_step": global_step, "train_loss": train_loss, **val_metrics}
    history.append(history_row)
    pd.DataFrame(history).to_csv(RUN_DIR / "metrics" / "history.csv", index=False)
    print(f"epoch {epoch}/{MAX_EPOCHS} loss={train_loss:.4f} val_bal_acc={current:.4f}")

    if current > best_metric + MIN_DELTA:
        best_metric = current
        best_epoch = epoch
        bad_epochs = 0
        best_adapter_state = {
            key: value.detach().cpu().clone()
            for key, value in get_peft_model_state_dict(model).items()
        }
        set_peft_model_state_dict(model, best_adapter_state)
        model.save_pretrained(RUN_DIR / "adapter")
        processor.save_pretrained(RUN_DIR / "adapter")
        print("Saved best adapter:", RUN_DIR / "adapter")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stopping at epoch {epoch}; best_epoch={best_epoch}")
            break

if best_adapter_state is not None:
    set_peft_model_state_dict(model, best_adapter_state)
else:
    model.save_pretrained(RUN_DIR / "adapter")
    processor.save_pretrained(RUN_DIR / "adapter")

print("Best epoch:", best_epoch)
print("Best val balanced_accuracy:", best_metric)


## 9. Evaluate Tiny-GenImage And CommunityForensics-Eval


In [ ]:
print("Evaluating Tiny-GenImage validation split for sanity check...")
tiny_metrics = evaluate_and_save(tiny_test_loader, dataset_tag="tiny_genimage_validation")

print("Evaluating CommunityForensics-Eval random streaming sample...")
commfor_loader = build_commfor_eval_loader()
commfor_metrics = evaluate_and_save(commfor_loader, dataset_tag="community_forensics_eval")

summary = {
    **config_summary,
    "best_epoch": best_epoch,
    "best_val_balanced_accuracy": best_metric,
    "adapter_path": str(RUN_DIR / "adapter"),
    "tiny_balanced_accuracy": tiny_metrics.get("balanced_accuracy"),
    "tiny_f1": tiny_metrics.get("f1"),
    "tiny_roc_auc": tiny_metrics.get("roc_auc"),
    "commfor_balanced_accuracy": commfor_metrics.get("balanced_accuracy"),
    "commfor_f1": commfor_metrics.get("f1"),
    "commfor_roc_auc": commfor_metrics.get("roc_auc"),
    "commfor_average_precision": commfor_metrics.get("average_precision"),
}
save_json(RUN_DIR / "metrics" / "summary.json", summary)

summary_df = pd.DataFrame([summary])
summary_path = OUTPUT_ROOT / f"summary_qwen25vl_lora_word_label_{RUN_ID}.csv"
manifest_path = OUTPUT_ROOT / f"adapter_manifest_{RUN_ID}.csv"
summary_df.to_csv(summary_path, index=False)
summary_df[[
    "run_id",
    "base_model_name",
    "adapter_path",
    "best_epoch",
    "best_val_balanced_accuracy",
    "tiny_balanced_accuracy",
    "tiny_roc_auc",
    "commfor_balanced_accuracy",
    "commfor_roc_auc",
    "commfor_average_precision",
]].to_csv(manifest_path, index=False)
print("Saved summary:", summary_path)
print("Saved adapter manifest:", manifest_path)
display(summary_df[[
    "adapter_path",
    "best_epoch",
    "best_val_balanced_accuracy",
    "tiny_balanced_accuracy",
    "tiny_roc_auc",
    "commfor_balanced_accuracy",
    "commfor_roc_auc",
    "commfor_average_precision",
]])


## 10. Zip Output For Download


In [ ]:
import shutil

zip_base = PROJECT_ROOT / f"qwen25vl_lora_word_label_{RUN_ID}"
shutil.make_archive(
    str(zip_base),
    "zip",
    RUN_DIR,
)
print("Created:", str(zip_base) + ".zip")


## 11. Load Adapter Later

Để dùng lại adapter trong notebook khác:

```python
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    device_map="auto",
    quantization_config=quantization_config,
)
model = PeftModel.from_pretrained(base_model, "/kaggle/input/<adapter-dataset>/adapter")
processor = AutoProcessor.from_pretrained("/kaggle/input/<adapter-dataset>/adapter")
```

Cần giữ lại folder `adapter/` và file `adapter_manifest_<run_id>.csv`.
